# MaduraApp — Fine-tuning YOLO26n en Kaggle

Pipeline CRISP-DM de entrenamiento del modelo de madurez agrícola.

**Antes de ejecutar cualquier celda:**
1. Panel derecho → **Accelerator → GPU P100** (requiere teléfono verificado)
2. Panel derecho → **Add Data** → subir `maduraapp_dataset.zip`
3. Anotar el nombre del dataset (ej: `maduraapp-dataset`) y ponerlo en la **Celda 3**

| Celda | Descripción | Tiempo aprox. |
|-------|-------------|---------------|
| 1 | Verificar GPU | < 1 min |
| 2 | Instalar dependencias | 2–3 min |
| 3 | Descomprimir dataset | 3–5 min |
| 4 | Verificar dataset | < 1 min |
| 5 | Generar data.yaml | < 1 min |
| 6 | **Entrenar (80 épocas)** | **1.5–2.5 horas** |
| 7 | Evaluar mAP\@50 | 5–10 min |
| 8 | Guardar best.pt en Output | < 1 min |
| 9 | Instrucciones para descargar | — |

## Celda 1 — Verificar GPU

In [ ]:
import torch

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU     : {gpu} ({mem:.1f} GB VRAM)')
else:
    raise SystemExit('Sin GPU — activa el acelerador en el panel derecho: Settings -> Accelerator -> GPU P100')

## Celda 2 — Instalar dependencias

In [ ]:
!pip install -q ultralytics pyyaml

from ultralytics import YOLO
import ultralytics
print(f'Ultralytics {ultralytics.__version__} instalado')

## Celda 3 — Descomprimir dataset

Cambia `DATASET_NAME` por el nombre exacto que Kaggle le asignó al ZIP que subiste.  
Lo encuentras en: panel derecho → **Data** → el nombre debajo del ícono del dataset.

In [ ]:
import subprocess, pathlib, time

# ── AJUSTAR: nombre del dataset tal como aparece en el panel Data ──────────
DATASET_NAME = 'maduraapp-dataset'   # <-- cambiar si es diferente
# ──────────────────────────────────────────────────────────────────────────

ZIP_PATH      = f'/kaggle/input/{DATASET_NAME}/maduraapp_dataset.zip'
LOCAL_DATASET = '/kaggle/working/maduraapp'

if not pathlib.Path(ZIP_PATH).exists():
    # Buscar el ZIP automáticamente si el nombre es distinto
    found = list(pathlib.Path('/kaggle/input').rglob('*.zip'))
    if found:
        ZIP_PATH = str(found[0])
        print(f'ZIP encontrado en: {ZIP_PATH}')
    else:
        raise FileNotFoundError(
            'No se encontró ningún ZIP en /kaggle/input.\n'
            'Asegúrate de haber añadido el dataset en el panel derecho -> Add Data.'
        )

zip_size = pathlib.Path(ZIP_PATH).stat().st_size / 1e9
print(f'ZIP encontrado: {zip_size:.2f} GB')

if not pathlib.Path(LOCAL_DATASET).exists():
    print('Descomprimiendo dataset...')
    t0 = time.time()
    subprocess.run(['unzip', '-q', ZIP_PATH, '-d', '/kaggle/working/'], check=True)
    print(f'Listo en {time.time() - t0:.0f}s')
else:
    print('Dataset ya existe, reutilizando.')

print(f'Dataset en: {LOCAL_DATASET}')

## Celda 4 — Verificar dataset

In [ ]:
import pathlib
from collections import Counter

CLASSES = [
    'aguacate_hass_INMADURO', 'aguacate_hass_OPTIMO', 'aguacate_hass_SOBRE_MADURO',
    'platano_INMADURO', 'platano_OPTIMO', 'platano_SOBRE_MADURO',
    'tomate_usda_INMADURO', 'tomate_usda_OPTIMO', 'tomate_usda_SOBRE_MADURO',
    'mango_INMADURO', 'mango_OPTIMO', 'mango_SOBRE_MADURO',
]

base = pathlib.Path(LOCAL_DATASET)
print(f'{"Split":<8}  {"Imágenes":>9}  {"Labels":>9}')
print('-' * 32)
total_imgs = 0
for split in ('train', 'valid', 'test'):
    imgs   = len(list((base / split / 'images').glob('*')))
    labels = len(list((base / split / 'labels').glob('*.txt')))
    print(f'{split:<8}  {imgs:>9,}  {labels:>9,}')
    total_imgs += imgs
print('-' * 32)
print(f'{"TOTAL":<8}  {total_imgs:>9,}')

# Distribución de bboxes por clase en train
counts = Counter()
for txt in (base / 'train' / 'labels').glob('*.txt'):
    for line in txt.read_text().splitlines():
        parts = line.strip().split()
        if parts:
            counts[int(parts[0])] += 1

print()
print(f'{"id":<4} {"clase":<35} {"bboxes":>8}')
print('-' * 50)
for cid, name in enumerate(CLASSES):
    n = counts.get(cid, 0)
    warn = ' <<' if n < 500 else ''
    print(f'{cid:<4} {name:<35} {n:>8,}{warn}')

## Celda 5 — Generar data.yaml

In [ ]:
import yaml

DATA_YAML = f'{LOCAL_DATASET}/data.yaml'

data_cfg = {
    'path':  LOCAL_DATASET,
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    'nc':    12,
    'names': CLASSES,
}

with open(DATA_YAML, 'w') as fh:
    yaml.dump(data_cfg, fh, allow_unicode=True)

print('data.yaml generado:')
print(open(DATA_YAML).read())

## Celda 6 — Entrenamiento YOLO26n (80 épocas)

Tiempo estimado: **1.5–2.5 horas** en GPU P100.  
Los checkpoints se guardan cada 10 épocas en `runs/maduraapp_v1/weights/`.  
Si la sesión se interrumpe, vuelve a ejecutar esta celda — reanuda automáticamente.

In [ ]:
from ultralytics import YOLO
import pathlib

RESUME_PT = pathlib.Path('/kaggle/working/runs/maduraapp_v1/weights/last.pt')

if RESUME_PT.exists():
    print(f'Checkpoint encontrado — reanudando desde {RESUME_PT}...')
    model = YOLO(str(RESUME_PT))
    results = model.train(resume=True)
else:
    print('Iniciando entrenamiento desde cero con YOLO26n...')
    model = YOLO('yolo26n.pt')
    results = model.train(
        data=DATA_YAML,
        epochs=80,
        batch=16,
        imgsz=640,
        device=0,
        optimizer='AdamW',
        lr0=0.001,
        lrf=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3.0,
        cos_lr=True,
        amp=True,
        patience=15,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        degrees=15.0,
        translate=0.1,
        scale=0.5,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.1,
        project='/kaggle/working/runs',
        name='maduraapp_v1',
        save_period=10,
        plots=True,
        exist_ok=True,
    )

BEST_PT = '/kaggle/working/runs/maduraapp_v1/weights/best.pt'
print(f'\nEntrenamiento finalizado. Modelo en: {BEST_PT}')

## Celda 7 — Evaluación sobre el set de test

KPI objetivo: **mAP\@50 ≥ 0.75**

In [ ]:
from ultralytics import YOLO

model   = YOLO(BEST_PT)
metrics = model.val(data=DATA_YAML, split='test', plots=True)

map50   = metrics.box.map50
map5095 = metrics.box.map
prec    = metrics.box.mp
recall  = metrics.box.mr

print()
print('=' * 45)
print(' RESULTADOS FINALES — MaduraApp v1')
print('=' * 45)
print(f'  mAP@50    = {map50:.4f}   (target >= 0.75)')
print(f'  mAP@50-95 = {map5095:.4f}')
print(f'  Precision = {prec:.4f}')
print(f'  Recall    = {recall:.4f}')
print()
if map50 >= 0.75:
    print('  KPI APROBADO -- modelo listo para deploy')
else:
    print(f'  KPI NO ALCANZADO (delta = {map50 - 0.75:.3f})')
    print('  Sugerencias:')
    print('    - Extender epocas: model.train(epochs=120, resume=True)')
    print('    - Reducir lr0 a 0.0005 y reentrenar')
print('=' * 45)

print()
print('mAP@50 por clase:')
for i, (name, ap) in enumerate(zip(CLASSES, metrics.box.ap50)):
    bar = '#' * int(ap * 20)
    print(f'  {i:>2} {name:<35} {ap:.3f} {bar}')

## Celda 8 — Guardar best.pt en Output

Kaggle guarda automáticamente todo lo que queda en `/kaggle/working/` al finalizar la sesión.  
Esta celda copia el modelo a la raíz del output para que sea fácil de encontrar.

In [ ]:
import shutil, pathlib

# Copiar best.pt a la raíz del output (más fácil de descargar)
dest = pathlib.Path('/kaggle/working/best.pt')
shutil.copy2(BEST_PT, dest)
print(f'best.pt copiado a {dest} ({dest.stat().st_size / 1e6:.1f} MB)')

# Copiar también la carpeta de resultados (curvas, confusion matrix)
results_src = pathlib.Path('/kaggle/working/runs/maduraapp_v1')
results_dst = pathlib.Path('/kaggle/working/maduraapp_results')
if not results_dst.exists():
    shutil.copytree(results_src, results_dst)
    print(f'Resultados copiados a {results_dst}')

print()
print('Para descargar best.pt:')
print('  Panel derecho -> Output -> best.pt -> icono de descarga')

## Celda 9 — Instrucciones para usar el modelo en el proyecto

Una vez descargado `best.pt` al PC, ejecuta en la raíz del proyecto:

In [ ]:
print('Pasos para integrar best.pt al backend:')
print()
print('1. Descargar best.pt desde panel Output de Kaggle')
print('2. Mover al proyecto:')
print('      mv ~/Downloads/best.pt runs/maduraapp_v1/weights/best.pt')
print()
print('3. Exportar al backend:')
print('      python scripts/export_model.py')
print()
print('4. Levantar el servidor:')
print('      cd backend && uvicorn app.main:app --reload')
print()
print(f'mAP@50 final del modelo: {map50:.4f}')

## (Opcional) Celda 10 — Visualizar predicciones sobre imágenes de test

In [ ]:
import glob, random
from IPython.display import Image, display
from ultralytics import YOLO

model     = YOLO(BEST_PT)
test_imgs = glob.glob(f'{LOCAL_DATASET}/test/images/*.jpg')
samples   = random.sample(test_imgs, min(5, len(test_imgs)))

for img_path in samples:
    result   = model(img_path)[0]
    out_path = result.save()
    print(img_path.split('/')[-1])
    display(Image(out_path))